In [6]:
import os
os.environ["GIT_PYTHON_REFRESH"] = "quiet"
#import mlflow
import pandas as pd

#mlflow.set_tracking_uri("sqlite:////home/felipe/mlflow.db")

#mlflow.set_experiment("Dataset-Moscow-Real-State")



# Imports

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats  # dependencia de statsmodels, ya viene instalada
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from scipy.stats import gaussian_kde, norm

pd.set_option("display.max_columns", None)

df_dpm = pd.read_csv("./MOSCOW-DATASET/district_prices_monthly.csv")
df_rentals = pd.read_csv("./MOSCOW-DATASET/rentals.csv")
df_secondary = pd.read_csv("./MOSCOW-DATASET/secondary_market.csv")
df_newbuilds = pd.read_csv("./MOSCOW-DATASET/new_builds.csv")
df_metro = pd.read_csv("./MOSCOW-DATASET/metro_stations.csv")

# Funciones para graficar

In [8]:
## Grafica variables categoricas como un grafico de barras horizontal mostrando la frecuencia de cada valor en porcentajes
def graficar_categoricos(df,variable,xlabel,title):
    fig, ax1 = plt.subplots(figsize=(6,10))
    categorias = (df[variable].value_counts(normalize=True) * 100).round(2)
    barras = ax1.barh(categorias.index,categorias.values,color="tab:blue")
    ax1.bar_label(barras, fmt="%.1f", padding=3)
    # valor al final de cada barra
    ax1.set_xlim(0, 100)
    # espacio para las etiquetas
    ax1.set_xlabel(xlabel)
    ax1.set_title(title)
    ax1.grid(axis="y", visible=False)
    # solo cuadricula vertical
    plt.show()
    return


def graficar_analisis_uni(variable,kde=True,densidad_normal=True):
    
    datos = variable.dropna()
    
    fig, ax1 = plt.subplot_mosaic([["hist","violin"],["box","box"]],figsize=(12, 7),
    width_ratios=[3, 2],                # columna izquierda 60%, derecha 40%
    height_ratios=[2, 1],               # fila de arriba el doble de alta
)
    conteos, bordes, _ = ax1["hist"].hist(variable,bins=30,color="steelblue",edgecolor="black")
    ax1["hist"].axvline(datos.mean(),color="red",linestyle="--",label=f"Media = {variable.mean():.1f}")
    x = np.linspace(datos.min(), datos.max(), 300)
    ancho = bordes[1] - bordes[0]
    media, desvio = datos.mean(), datos.std()
    curva = gaussian_kde(datos)(x) * len(datos) * ancho
    escala = len(datos) * ancho
    
    if kde:
        ax1["hist"].plot(x, curva, color="darkorange",
                lw=2, label="KDE")
    if densidad_normal:
        mu, sigma = norm.fit(datos)                  # parametros de la normal ajustada
        ax1["hist"].plot(x, norm.pdf(x, mu, sigma) * escala,
                 color="turquoise", linestyle="--", linewidth=2,
                 label=f"Normal ajustada (mu={mu:,.0f}, sigma={sigma:,.0f})")
        
    ax1["hist"].set_title(f"Distribucion de {variable.name}")
    ax1["hist"].set_xlabel(f"{variable.name}")
    ax1["hist"].set_ylabel("Frecuencia")
    ax1["hist"].legend()

    ax1["violin"].violinplot(datos,showmedians=True)
    ax1["violin"].set_title(f"Violin-Plot de {variable.name}")
    
    ax1["box"].boxplot(datos,orientation='horizontal',patch_artist=True,showmeans=True)
    ax1["box"].set_title(f"Box-Plot de {variable.name}")
    
    plt.tight_layout()
    plt.show()

    
    return

def graficar_boxplot(variable):
    
    fig, ax = plt.subplots(figsize = (5,5))
    ax.boxplot(variable, patch_artist=True ,showmeans = True)
    ax.set_title(f"Box-Plot de {variable.name}") 
    plt.show()
    return 

def graficar_binarias(df):
    bin_cols = [c for c in df.columns if df[c].nunique() == 2]
    prev = (df[bin_cols].mean() * 100).sort_values()   # media de 0/1 = proporcion de 1
    fig, ax = plt.subplots(figsize=(6, 0.55 * len(prev) + 1.5))   # alto segun cantidad
    barras = ax.barh(prev.index, prev.values, color="tab:blue")
    ax.bar_label(barras, fmt="%.1f%%", padding=3)
    ax.set_xlim(0, 100)                             # escala completa 0-100%
    ax.set_xlabel("% de casos positivos")
    ax.set_title("Prevalencia de cada variable")
    ax.grid(axis="y", visible=False)
    plt.show()
    return

# Descripcion de las columnas

## rentals.csv: anuncios de alquiler (20 000 filas)

| Columna | Descripción |
|---|---|
| `id` | ID único del anuncio |
| `date_posted` | Fecha de publicación |
| `district` | Distrito (район) |
| `okrug` | Okrug administrativo (11 en total; `TAO`, `NAO` y `ZelAO` corresponden a Nueva Moscú) |
| `lat`, `lon` | Coordenadas del inmueble |
| `total_area` | Superficie total, m² |
| `rooms` | Número de habitaciones (1–4) |
| `floor` | Piso |
| `total_floors` | Número de pisos del edificio |
| `building_year` | Año de construcción del edificio |
| `building_type` | Tipo de edificio: `brick` (ladrillo), `monolith` (monolítico), `panel` (paneles), `modern_panel` (panel moderno), `stalin` (stalinka), `khrushchev` (jrushchovka) |
| `renovation` | Reforma: `cosmetic` (cosmética), `euro` (euro-reforma), `designer` (de diseño), `no_renovation` (sin reforma) |
| `furnished` | Indica si tiene muebles (bool) |
| `pets_allowed` | Indica si se admiten mascotas (bool) |
| `deposit_months` | Depósito, expresado en meses de alquiler (0–2) |
| `metro_station` | Estación de metro más cercana |
| `metro_line` | Línea de metro |
| `metro_distance_min` | Tiempo hasta el metro, minutos |
| `to_center_km` | Distancia al centro de la ciudad, km |
| `monthly_rent_rub` | Alquiler mensual, rublos |
| `rent_per_sqm` | Alquiler mensual por m², rublos (variable principal para el análisis) |

## secondary_market.csv: vivienda de segunda mano en venta (50 000 filas)

| Columna | Descripción |
|---|---|
| `id` | ID único del anuncio |
| `date_posted` | Fecha de publicación |
| `district` | Distrito (район) |
| `okrug` | Okrug administrativo |
| `lat`, `lon` | Coordenadas del inmueble |
| `total_area` | Superficie total, m² |
| `living_area` | Superficie habitable, m² |
| `kitchen_area` | Superficie de la cocina, m² |
| `rooms` | Número de habitaciones |
| `floor` | Piso |
| `total_floors` | Número de pisos del edificio |
| `building_year` | Año de construcción del edificio |
| `building_type` | Tipo de edificio (mismos valores que en `rentals`) |
| `ceiling_height` | Altura de techos, m |
| `has_balcony` | Indica si tiene balcón o logia (bool) |
| `renovation` | Reforma (mismos valores que en `rentals`) |
| `metro_station` | Estación de metro más cercana |
| `metro_line` | Línea de metro |
| `metro_distance_min` | Tiempo hasta el metro, minutos |
| `metro_distance_type` | Cómo se calculó el tiempo al metro: `walk` (a pie) o `transport` (en transporte) |
| `to_center_km` | Distancia al centro de la ciudad, km |
| `price_rub` | Precio de venta, rublos |
| `price_per_sqm` | Precio de venta por m², rublos (variable principal para el análisis) |
| `mortgage_rate_at_listing` | Tasa hipotecaria en el momento de publicar el anuncio, % |
| `seller_type` | Tipo de vendedor: `agency` (agencia) o `owner` (propietario) |

## new_builds.csv: obra nueva en venta (8 000 filas)

| Columna | Descripción |
|---|---|
| `id` | ID único del anuncio |
| `date_posted` | Fecha de publicación |
| `complex_id` | ID del complejo residencial |
| `complex_name` | Nombre del complejo residencial |
| `developer` | Promotora o constructora |
| `complex_class` | Clase de vivienda: `economy`, `comfort`, `business`, `premium` |
| `district` | Distrito (район) |
| `okrug` | Okrug administrativo |
| `lat`, `lon` | Coordenadas del inmueble |
| `total_area` | Superficie total, m² |
| `rooms` | Número de habitaciones |
| `floor` | Piso |
| `total_floors` | Número de pisos del edificio |
| `completion_year` | Año de entrega del edificio |
| `ready_status` | Estado: `under_construction` (en construcción) o `completed` (entregado) |
| `metro_station` | Estación de metro más cercana |
| `metro_line` | Línea de metro |
| `metro_distance_min` | Tiempo hasta el metro, minutos |
| `to_center_km` | Distancia al centro de la ciudad, km |
| `price_rub` | Precio de venta, rublos |
| `price_per_sqm` | Precio de venta por m², rublos (variable principal para el análisis) |
| `deal_type` | Tipo de operación: `DDU`/`FZ214` (contrato de participación en la construcción), `installment` (pago a plazos), `trade-in` (entrega de vivienda antigua como parte de pago) |
| `subsidized_mortgage` | Indica si hay hipoteca subsidiada disponible (bool) |
| `mortgage_rate_at_listing` | Tasa hipotecaria en el momento de publicar el anuncio, % |

## district_prices_monthly.csv: estadística mensual por distrito (9 804 filas)

Fuente independiente de los tres archivos de anuncios; sirve como referencia para contrastar cálculos propios.

| Columna | Descripción |
|---|---|
| `year_month` | Mes (primer día del mes) |
| `district` | Distrito (район) |
| `okrug` | Okrug administrativo |
| `secondary_price_per_sqm` | Precio medio oficial de segunda mano por m² |
| `newbuild_price_per_sqm` | Precio medio oficial de obra nueva por m² |
| `rental_price_per_sqm_monthly` | Alquiler medio oficial por m² al mes |
| `secondary_mom_change_pct` | Variación del precio de segunda mano respecto al mes anterior, % (month-over-month) |
| `newbuild_mom_change_pct` | Ídem para obra nueva. Ambas columnas `mom_change_pct` están vacías en 2020-01 porque es el primer mes y no hay periodo previo |
| `n_listings_secondary` | Número de anuncios de segunda mano usados para calcular la estadística |
| `n_listings_newbuild` | Número de anuncios de obra nueva usados para calcular la estadística |
| `n_listings_rental` | Número de anuncios de alquiler usados para calcular la estadística |
| `cbr_key_rate_pct` | Tasa clave del Banco Central de Rusia en ese mes, % |
| `avg_mortgage_rate_pct` | Tasa hipotecaria media en ese mes, % |

## metro_stations.csv: catálogo de estaciones de metro (104 filas)

Se une con los demás archivos mediante `metro_station` = `station_name`.

| Columna | Descripción |
|---|---|
| `station_name` | Nombre de la estación |
| `line` | Línea de metro |
| `lat`, `lon` | Coordenadas de la estación |
| `year_opened` | Año de apertura de la estación |
| `to_center_km` | Distancia de la estación al centro de la ciudad, km |

# Describe e info 

## District prices monthly

In [9]:
df_dpm.describe()

,secondary_price_per_sqm,newbuild_price_per_sqm,rental_price_per_sqm_monthly,secondary_mom_change_pct,newbuild_mom_change_pct,n_listings_secondary,n_listings_newbuild,n_listings_rental,cbr_key_rate_pct,avg_mortgage_rate_pct
count,9804.000000,9804.000000,9804.000000,9675.000000,9675.000000,9804.000000,9804.000000,9804.000000,9804.000000,9804.000000
mean,293669.543044,323147.460220,1223.625051,0.653792,0.713295,109.478070,42.461444,64.638821,11.986842,14.486842
std,123593.449575,136415.279338,514.966664,3.693047,5.149745,52.000534,21.655502,31.757889,5.712394,5.712394
min,70600.000000,74700.000000,294.000000,-13.090000,-15.630000,20.000000,5.000000,10.000000,4.250000,6.750000
25%,220400.000000,242375.000000,918.000000,-1.890000,-2.820000,64.000000,24.000000,38.000000,7.500000,10.000000
50%,274200.000000,301000.000000,1142.500000,0.600000,0.520000,110.000000,43.000000,65.000000,9.500000,12.000000
75%,344600.000000,379000.000000,1436.000000,3.130000,4.150000,155.000000,61.000000,92.000000,17.000000,19.500000
max,812700.000000,924900.000000,3386.000000,15.740000,25.910000,199.000000,79.000000,119.000000,21.000000,23.500000


In [10]:
df_dpm.info()

<class 'pandas.DataFrame'>
RangeIndex: 9804 entries, 0 to 9803
Data columns (total 13 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   year_month                    9804 non-null   str    
 1   district                      9804 non-null   str    
 2   okrug                         9804 non-null   str    
 3   secondary_price_per_sqm       9804 non-null   int64  
 4   newbuild_price_per_sqm        9804 non-null   int64  
 5   rental_price_per_sqm_monthly  9804 non-null   int64  
 6   secondary_mom_change_pct      9675 non-null   float64
 7   newbuild_mom_change_pct       9675 non-null   float64
 8   n_listings_secondary          9804 non-null   int64  
 9   n_listings_newbuild           9804 non-null   int64  
 10  n_listings_rental             9804 non-null   int64  
 11  cbr_key_rate_pct              9804 non-null   float64
 12  avg_mortgage_rate_pct         9804 non-null   float64
dtypes: float64(4),

In [11]:
df_dpm.nunique()

year_month                        76
district                         129
okrug                             11
secondary_price_per_sqm         3886
newbuild_price_per_sqm          4088
rental_price_per_sqm_monthly    2154
secondary_mom_change_pct        1737
newbuild_mom_change_pct         2259
n_listings_secondary             180
n_listings_newbuild               75
n_listings_rental                110
cbr_key_rate_pct                  10
avg_mortgage_rate_pct             10
dtype: int64

## Rentals

In [12]:
df_rentals.describe()

,lat,lon,total_area,rooms,floor,total_floors,building_year,deposit_months,metro_distance_min,to_center_km,monthly_rent_rub,rent_per_sqm
count,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000
mean,55.712537,37.542166,55.387365,1.794150,6.608400,12.190750,1987.822600,1.101450,152.385200,16.360104,52432.855000,945.461200
std,0.144287,0.180773,12.739508,0.865456,4.829805,5.691844,22.499783,0.540715,167.803358,12.800552,30003.240405,480.537062
min,55.229580,36.702390,22.100000,1.000000,1.000000,5.000000,1935.000000,0.000000,0.000000,0.020000,6800.000000,148.000000
25%,55.635830,37.464980,45.700000,1.000000,3.000000,9.000000,1968.000000,1.000000,33.000000,9.040000,32800.000000,644.000000
50%,55.740440,37.563905,53.800000,2.000000,5.000000,9.000000,1988.000000,1.000000,103.000000,13.230000,45800.000000,845.000000
75%,55.817793,37.652073,63.300000,2.000000,9.000000,17.000000,2008.000000,1.000000,198.000000,18.030000,64400.000000,1135.000000
max,56.012530,37.908100,104.600000,4.000000,22.000000,22.000000,2024.000000,2.000000,966.000000,71.060000,300700.000000,4264.000000


In [13]:
df_rentals.info()
df_rentals.nunique()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 22 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  20000 non-null  str    
 1   date_posted         20000 non-null  str    
 2   district            20000 non-null  str    
 3   okrug               20000 non-null  str    
 4   lat                 20000 non-null  float64
 5   lon                 20000 non-null  float64
 6   total_area          20000 non-null  float64
 7   rooms               20000 non-null  int64  
 8   floor               20000 non-null  int64  
 9   total_floors        20000 non-null  int64  
 10  building_year       20000 non-null  int64  
 11  building_type       20000 non-null  str    
 12  renovation          20000 non-null  str    
 13  furnished           20000 non-null  bool   
 14  pets_allowed        20000 non-null  bool   
 15  deposit_months      20000 non-null  int64  
 16  metro_station  

id                    20000
date_posted            1945
district                129
okrug                    11
lat                   15985
lon                   16678
total_area              695
rooms                     4
floor                    22
total_floors              5
building_year            90
building_type             6
renovation                4
furnished                 2
pets_allowed              2
deposit_months            3
metro_station           104
metro_line               12
metro_distance_min      852
to_center_km           4266
monthly_rent_rub       1576
rent_per_sqm           2335
dtype: int64

In [14]:
df_rentals.head()

,id,date_posted,district,okrug,lat,lon,total_area,rooms,floor,total_floors,building_year,building_type,renovation,furnished,pets_allowed,deposit_months,metro_station,metro_line,metro_distance_min,to_center_km,monthly_rent_rub,rent_per_sqm
0,R000001,2025-01-16,Nagatinsky Zaton,YuAO,55.70058,37.69344,34.7,1,5,9,1958,brick,cosmetic,False,True,1,Ugreshskaya,MCC,28,7.47,26200,756
1,R000002,2022-11-12,Chertanovo Severnoye,YuAO,55.64717,37.61690,53.4,1,10,12,2010,monolith,cosmetic,True,False,1,Kuznetsky Most,Tagansko-Krasnopresnenskaya,152,11.87,37200,696
2,R000003,2026-03-06,Savyolovsky,SAO,55.78584,37.57104,69.9,2,15,22,2009,monolith,euro,True,False,1,Rizhskaya,Kaluzhsko-Rizhskaya,43,4.72,96700,1384
3,R000004,2024-10-11,Beskudnikovsky,SAO,55.87034,37.56080,44.6,2,14,17,1980,brick,no_renovation,True,False,1,Trubnaya,Lyublinsko-Dmitrovskaya,145,13.48,30200,677
4,R000005,2023-07-11,Shchapovskoye,TAO,55.37723,37.51217,56.7,1,3,5,1979,panel,euro,False,False,1,Dubrovka,Lyublinsko-Dmitrovskaya,472,42.44,18700,330


## Secondary market

In [15]:
df_secondary.describe()

,lat,lon,total_area,living_area,kitchen_area,rooms,floor,total_floors,building_year,ceiling_height,metro_distance_min,to_center_km,price_rub,price_per_sqm,mortgage_rate_at_listing
count,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.00000,50000.00000,50000.000000,50000.000000,50000.000000,5.000000e+04,5.000000e+04,50000.000000
mean,55.712683,37.543657,67.053158,41.883782,11.244650,2.134900,7.475420,14.00086,1988.22008,2.787734,57.063900,16.326353,1.312282e+07,1.956257e+05,14.478795
std,0.143998,0.180338,17.309579,11.209233,2.855538,1.023153,5.530469,6.53193,23.57096,0.220836,54.483163,12.737610,8.564073e+06,1.132709e+05,5.702273
min,55.234080,36.699230,20.900000,12.600000,6.000000,1.000000,1.000000,5.00000,1935.00000,2.500000,2.000000,0.070000,1.019000e+06,2.340000e+04,6.750000
25%,55.635415,37.467110,54.400000,33.700000,9.200000,1.000000,3.000000,9.00000,1968.00000,2.700000,22.000000,9.027500,7.481750e+06,1.230000e+05,10.000000
50%,55.740110,37.565840,65.100000,40.600000,11.100000,2.000000,6.000000,12.00000,1989.00000,2.700000,39.000000,13.230000,1.108600e+07,1.704000e+05,12.000000
75%,55.817490,37.652020,77.500000,48.500000,13.100000,3.000000,10.000000,22.00000,2009.00000,3.000000,71.000000,18.060000,1.638725e+07,2.395000e+05,19.500000
max,56.020470,37.919860,134.300000,89.200000,24.800000,5.000000,25.000000,25.00000,2024.00000,3.500000,328.000000,70.550000,9.343700e+07,1.239100e+06,23.500000


In [16]:
df_secondary.info()


<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 26 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   id                        50000 non-null  str    
 1   date_posted               50000 non-null  str    
 2   district                  50000 non-null  str    
 3   okrug                     50000 non-null  str    
 4   lat                       50000 non-null  float64
 5   lon                       50000 non-null  float64
 6   total_area                50000 non-null  float64
 7   living_area               50000 non-null  float64
 8   kitchen_area              50000 non-null  float64
 9   rooms                     50000 non-null  int64  
 10  floor                     50000 non-null  int64  
 11  total_floors              50000 non-null  int64  
 12  building_year             50000 non-null  int64  
 13  building_type             50000 non-null  str    
 14  ceiling_height   

In [17]:
df_secondary.nunique()

id                          50000
date_posted                  2311
district                      129
okrug                          11
lat                         30403
lon                         33092
total_area                   1029
living_area                   702
kitchen_area                  170
rooms                           5
floor                          25
total_floors                    6
building_year                  90
building_type                   6
ceiling_height                  6
has_balcony                     2
renovation                      4
metro_station                 104
metro_line                     12
metro_distance_min            317
metro_distance_type             2
to_center_km                 5550
price_rub                   21983
price_per_sqm                5468
mortgage_rate_at_listing       10
seller_type                     2
dtype: int64

## New Builds

In [18]:
df_newbuilds.describe()

,lat,lon,total_area,rooms,floor,total_floors,completion_year,metro_distance_min,to_center_km,price_rub,price_per_sqm,mortgage_rate_at_listing
count,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8.000000e+03,8000.000000,8000.000000
mean,55.713163,37.529800,55.256312,1.691625,12.850625,24.716625,2025.937125,169.012875,17.414789,1.418128e+07,256610.950000,15.906156
std,0.151999,0.210704,17.282473,0.998075,7.754262,4.984644,1.089986,187.256063,14.501530,8.546752e+06,126395.896256,5.188344
min,55.268010,36.710280,22.000000,0.000000,1.000000,17.000000,2024.000000,1.000000,0.360000,1.371000e+06,52700.000000,6.750000
25%,55.632790,37.451500,42.800000,1.000000,6.000000,22.000000,2025.000000,38.000000,8.990000,8.054750e+06,171775.000000,11.000000
50%,55.754310,37.558990,54.300000,2.000000,12.000000,25.000000,2026.000000,107.000000,13.690000,1.241000e+07,229650.000000,15.500000
75%,55.819700,37.654990,66.400000,2.000000,19.000000,30.000000,2027.000000,212.000000,18.990000,1.799200e+07,318625.000000,19.500000
max,56.006140,37.874520,112.200000,4.000000,35.000000,35.000000,2028.000000,960.000000,70.220000,7.908300e+07,813800.000000,23.500000


In [19]:
df_newbuilds.info()

<class 'pandas.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 25 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   id                        8000 non-null   str    
 1   date_posted               8000 non-null   str    
 2   complex_id                8000 non-null   str    
 3   complex_name              8000 non-null   str    
 4   developer                 8000 non-null   str    
 5   complex_class             8000 non-null   str    
 6   district                  8000 non-null   str    
 7   okrug                     8000 non-null   str    
 8   lat                       8000 non-null   float64
 9   lon                       8000 non-null   float64
 10  total_area                8000 non-null   float64
 11  rooms                     8000 non-null   int64  
 12  floor                     8000 non-null   int64  
 13  total_floors              8000 non-null   int64  
 14  completion_year    

In [20]:
df_newbuilds.nunique()

id                          8000
date_posted                 1910
complex_id                   250
complex_name                  20
developer                     11
complex_class                  4
district                     109
okrug                         11
lat                          249
lon                          250
total_area                   803
rooms                          5
floor                         35
total_floors                   5
completion_year                5
ready_status                   2
metro_station                104
metro_line                    12
metro_distance_min           804
to_center_km                 241
price_rub                   6836
price_per_sqm               3673
deal_type                      4
subsidized_mortgage            2
mortgage_rate_at_listing       9
dtype: int64

## Metro stations

In [21]:
df_metro.describe()

,lat,lon,year_opened,to_center_km
count,104.000000,104.000000,104.000000,104.000000
mean,55.746107,37.603331,1970.317308,6.290385
std,0.050591,0.083541,22.486788,4.580953
min,55.602800,37.355900,1935.000000,0.330000
25%,55.728575,37.551950,1953.750000,2.512500
50%,55.752150,37.606350,1967.500000,5.420000
75%,55.776000,37.642225,1986.000000,9.222500
max,55.855100,37.813300,2018.000000,20.020000


In [22]:
df_metro.info()

<class 'pandas.DataFrame'>
RangeIndex: 104 entries, 0 to 103
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   station_name  104 non-null    str    
 1   line          104 non-null    str    
 2   lat           104 non-null    float64
 3   lon           104 non-null    float64
 4   year_opened   104 non-null    int64  
 5   to_center_km  104 non-null    float64
dtypes: float64(3), int64(1), str(2)
memory usage: 8.1 KB


In [23]:
df_metro.nunique()

station_name    104
line             12
lat              98
lon              98
year_opened      44
to_center_km    103
dtype: int64

# Analisis Bivariado

**Objetivo:** precio por m² (`rent_per_sqm` / `price_per_sqm`).

**Orden:** Funciones → por dataset: Nivel 1 (todas×todas) → correlaciones numéricas → detalle **todas** las features vs objetivo → num×cat → cat×cat.

Cada bloque tiene titulo `###` y su celda de codigo.


## Funciones — analisis bivariado

Nivel 1: matriz de asociacion mixta (todos los pares) + ranking de pares.

Nivel 2: por tipo de combinacion — grafico principal, complementos y prueba estadistica (segun enunciado).


In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except (NameError, AttributeError):
    pass

import seaborn as sns
from scipy.stats import (
    chi2_contingency,
    spearmanr,
    pearsonr,
    mannwhitneyu,
    kruskal,
    f_oneway,
    pointbiserialr,
)

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")

EXCLUIR_BIVARIADO = {
    "id",
    "date_posted",
    "complex_id",
    "monthly_rent_rub",
    "price_rub",
}

MAX_FILAS_GRAFICO = 8000
MAX_CAT_GRAFICO = 12
TOP_PARES_CAT_CAT = 8


def _muestra_plot(df):
    if len(df) > MAX_FILAS_GRAFICO:
        return df.sample(MAX_FILAS_GRAFICO, random_state=42)
    return df


def cramers_v(x, y):
    tabla = pd.crosstab(x, y)
    if tabla.size == 0 or tabla.shape[0] < 2 or tabla.shape[1] < 2:
        return np.nan
    chi2, _, _, _ = chi2_contingency(tabla)
    n = tabla.to_numpy().sum()
    r, k = tabla.shape
    denom = n * min(r - 1, k - 1)
    if denom <= 0:
        return np.nan
    return float(np.sqrt(chi2 / denom))


def eta_squared(categorica, numerica):
    cat = pd.Categorical(categorica)
    y = pd.to_numeric(numerica, errors="coerce")
    mask = pd.notna(y) & pd.notna(categorica)
    y = y[mask]
    cat = cat[mask]
    if len(y) < 3 or cat.categories.size < 2:
        return np.nan
    grupos = [y[cat == c].values for c in cat.categories]
    grupos = [g for g in grupos if len(g) > 0]
    if len(grupos) < 2:
        return np.nan
    try:
        f_oneway(*grupos)
    except ValueError:
        return np.nan
    ss_between = sum(len(g) * (g.mean() - y.mean()) ** 2 for g in grupos)
    ss_total = ((y - y.mean()) ** 2).sum()
    if ss_total == 0:
        return np.nan
    return float(ss_between / ss_total)


def clasificar_tipo(serie, nombre, umbral_discreto=20):
    s = serie.dropna()
    if len(s) == 0:
        return "desconocido"
    if pd.api.types.is_bool_dtype(serie) or s.nunique() == 2:
        return "binaria"
    if pd.api.types.is_numeric_dtype(serie):
        if s.nunique() <= umbral_discreto:
            return "numerica_discreta"
        return "numerica_continua"
    return "categorica"


def _serie_binaria(serie):
    if pd.api.types.is_bool_dtype(serie):
        return serie.astype(int)
    return pd.Series(pd.Categorical(serie).codes, index=serie.index)


def medida_asociacion(df, col_a, col_b, tipo_a, tipo_b):
    sub = df[[col_a, col_b]].dropna()
    if len(sub) < 5:
        return np.nan
    a, b = sub[col_a], sub[col_b]

    def es_num(t):
        return t in ("numerica_continua", "numerica_discreta")

    if es_num(tipo_a) and es_num(tipo_b):
        if sub[col_a].nunique() < 2 or sub[col_b].nunique() < 2:
            return np.nan
        r, _ = spearmanr(sub[col_a], sub[col_b])
        return float(abs(r))

    if tipo_a == "binaria" and tipo_b == "binaria":
        x, y = _serie_binaria(a), _serie_binaria(b)
        if x.nunique() < 2 or y.nunique() < 2:
            return np.nan
        r, _ = pearsonr(x, y)
        return float(abs(r))

    if tipo_a == "binaria" and es_num(tipo_b):
        x = _serie_binaria(a)
        if x.nunique() < 2:
            return np.nan
        r, _ = pointbiserialr(x, sub[col_b])
        return float(abs(r))
    if tipo_b == "binaria" and es_num(tipo_a):
        x = _serie_binaria(b)
        if x.nunique() < 2:
            return np.nan
        r, _ = pointbiserialr(x, sub[col_a])
        return float(abs(r))

    if es_num(tipo_a) and tipo_b in ("categorica", "numerica_discreta"):
        return float(np.sqrt(eta_squared(b, a)))
    if es_num(tipo_b) and tipo_a in ("categorica", "numerica_discreta"):
        return float(np.sqrt(eta_squared(a, b)))

    if tipo_a in ("categorica", "numerica_discreta") and tipo_b in (
        "categorica",
        "numerica_discreta",
    ):
        return cramers_v(a, b)

    return np.nan


def matriz_asociacion_mixta(df, columnas, tipos):
    mat = pd.DataFrame(np.nan, index=columnas, columns=columnas)
    for i, c1 in enumerate(columnas):
        mat.loc[c1, c1] = 1.0
        for c2 in columnas[i + 1 :]:
            v = medida_asociacion(df, c1, c2, tipos[c1], tipos[c2])
            mat.loc[c1, c2] = v
            mat.loc[c2, c1] = v
    return mat


def ranking_pares_vs_target(mat, target, top_n=20):
    filas = []
    for col in mat.columns:
        if col == target:
            continue
        filas.append({"feature": col, "asociacion": mat.loc[target, col]})
    return pd.DataFrame(filas).sort_values("asociacion", ascending=False).head(top_n)


def ranking_pares_globales(mat, top_n=30):
    filas = []
    cols = list(mat.columns)
    for i, c1 in enumerate(cols):
        for c2 in cols[i + 1 :]:
            v = mat.loc[c1, c2]
            if pd.notna(v):
                filas.append({"var_a": c1, "var_b": c2, "asociacion": v})
    out = pd.DataFrame(filas).sort_values("asociacion", ascending=False)
    return out.head(top_n)


def _top_categorias(serie, max_cat=MAX_CAT_GRAFICO):
    return serie.value_counts().head(max_cat).index.tolist()


def preparar_bivariado(df, target, nombre_dataset):
    print(f"Dataset: {nombre_dataset} | objetivo: {target}")
    columnas = [c for c in df.columns if c not in EXCLUIR_BIVARIADO and c != target]
    columnas_analisis = [target] + columnas
    tipos = {c: clasificar_tipo(df[c], c) for c in columnas_analisis}
    mat = matriz_asociacion_mixta(df, columnas_analisis, tipos)
    rank_target = ranking_pares_vs_target(mat, target, top_n=len(columnas))
    rank_pares = ranking_pares_globales(mat, top_n=min(30, len(columnas) * 2))
    return mat, rank_target, rank_pares, tipos, columnas


def nivel1_todas_por_todas(mat, rank_pares, rank_target, target, nombre_dataset):
    print("--- Nivel 1: asociacion mixta (todas x todas) ---")
    fig, ax = plt.subplots(figsize=(max(10, len(mat) * 0.45), max(8, len(mat) * 0.4)))
    sns.heatmap(mat, annot=False, cmap="YlOrRd", vmin=0, vmax=1, ax=ax)
    ax.set_title(f"{nombre_dataset} — heatmap asociacion mixta")
    plt.tight_layout()
    plt.show()

    print("Ranking — pares mas asociados (global):")
    try:
        display(rank_pares)
    except NameError:
        print(rank_pares.to_string(index=False))

    fig, ax = plt.subplots(figsize=(8, max(4, 0.35 * len(rank_pares))))
    labels = rank_pares["var_a"] + " × " + rank_pares["var_b"]
    sns.barplot(x=rank_pares["asociacion"], y=labels, ax=ax, palette="viridis")
    ax.set_xlim(0, 1)
    ax.set_xlabel("Asociacion (0-1)")
    ax.set_title(f"{nombre_dataset} — top pares asociados")
    plt.tight_layout()
    plt.show()

    print(f"Ranking — variables vs {target}:")
    try:
        display(rank_target)
    except NameError:
        print(rank_target.to_string(index=False))


def nivel1_correlaciones_numericas(df, tipos, nombre_dataset):
    nums = [
        c for c in tipos
        if tipos[c] in ("numerica_continua", "numerica_discreta")
    ]
    sub = df[nums].dropna()
    if len(nums) < 2 or len(sub) < 5:
        print("Pocas columnas numericas para correlacion.")
        return nums
    for metodo, nombre in [("pearson", "Pearson"), ("spearman", "Spearman")]:
        corr = sub.corr(method=metodo)
        fig, ax = plt.subplots(figsize=(max(8, len(nums) * 0.5), max(6, len(nums) * 0.45)))
        sns.heatmap(corr, annot=len(nums) <= 12, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
        ax.set_title(f"{nombre_dataset} — {nombre}")
        plt.tight_layout()
        plt.show()
    return nums


def detalle_numerica_continua_vs_target(df, x_col, y_col, titulo):
    sub = _muestra_plot(df[[x_col, y_col]].dropna())
    if len(sub) < 10:
        return
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.regplot(data=sub, x=x_col, y=y_col, scatter_kws={"alpha": 0.25, "s": 12}, ax=ax)
    ax.set_title(f"{titulo}: {y_col} vs {x_col}")
    plt.tight_layout()
    plt.show()

    r_p, p_p = pearsonr(sub[x_col], sub[y_col])
    r_s, p_s = spearmanr(sub[x_col], sub[y_col])
    print(f"  Pearson r={r_p:.3f} (p={p_p:.3g}) | Spearman rho={r_s:.3f} (p={p_s:.3g})")

    g = sns.jointplot(data=sub, x=x_col, y=y_col, kind="scatter", height=5, marginal_ticks=True)
    g.fig.suptitle(f"Jointplot — {x_col} vs {y_col}", y=1.02)
    plt.show()

    fig, ax = plt.subplots(figsize=(7, 5))
    hb = ax.hexbin(sub[x_col], sub[y_col], gridsize=35, cmap="YlOrRd", mincnt=1)
    plt.colorbar(hb, ax=ax, label="conteo")
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.set_title(f"Hexbin — {y_col} vs {x_col}")
    plt.tight_layout()
    plt.show()


def pairplot_numericas(df, cols_num, hue_col, titulo, max_cols=7):
    cols = [c for c in cols_num if c in df.columns][:max_cols]
    if len(cols) < 2:
        return
    extra = [hue_col] if hue_col in df.columns else []
    sub = _muestra_plot(df[cols + extra].dropna())
    if hue_col in sub.columns and sub[hue_col].nunique() > 8:
        sub = sub.copy()
        sub[hue_col] = sub[hue_col].astype(str)
        sub = sub[sub[hue_col].isin(_top_categorias(sub[hue_col], 6))]
    print(f"  Pairplot — columnas: {cols}")
    g = sns.pairplot(
        sub,
        vars=cols,
        hue=hue_col if hue_col in sub.columns else None,
        corner=True,
        plot_kws={"alpha": 0.3, "s": 10},
    )
    g.fig.suptitle(titulo, y=1.02)
    plt.show()


def detalle_numerica_vs_categorica(df, num_col, cat_col, titulo, max_cat=MAX_CAT_GRAFICO):
    sub = df[[num_col, cat_col]].dropna()
    if len(sub) < 10:
        return
    if sub[cat_col].nunique() > max_cat:
        sub = sub[sub[cat_col].isin(_top_categorias(sub[cat_col], max_cat))]
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.boxplot(data=sub, x=cat_col, y=num_col, ax=ax)
    sns.stripplot(data=sub, x=cat_col, y=num_col, color="black", alpha=0.15, size=2, ax=ax)
    ax.set_title(f"{titulo}: {num_col} por {cat_col}")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.violinplot(data=sub, x=cat_col, y=num_col, inner="quartile", ax=ax)
    ax.set_title(f"Violin — {num_col} por {cat_col}")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

    grupos = [g[num_col].values for _, g in sub.groupby(cat_col)]
    grupos = [g for g in grupos if len(g) > 0]
    if len(grupos) >= 2:
        stat, p = kruskal(*grupos)
        print(f"  Kruskal-Wallis H={stat:.3f} (p={p:.3g})")


def detalle_categorica_vs_target(df, cat_col, target, titulo, max_cat=MAX_CAT_GRAFICO):
    sub = df[[cat_col, target]].dropna()
    if len(sub) < 10:
        return
    if sub[cat_col].nunique() > max_cat:
        sub = sub[sub[cat_col].isin(_top_categorias(sub[cat_col], max_cat))]
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.boxplot(data=sub, x=cat_col, y=target, ax=ax)
    sns.stripplot(data=sub, x=cat_col, y=target, color="black", alpha=0.15, size=2, ax=ax)
    ax.set_title(f"{titulo}: {target} por {cat_col}")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.violinplot(data=sub, x=cat_col, y=target, inner="quartile", ax=ax)
    ax.set_title(f"Violin — {target} por {cat_col}")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(data=sub, x=cat_col, y=target, estimator="mean", errorbar=("ci", 95), ax=ax)
    ax.set_title(f"Media de {target} con IC 95% por {cat_col}")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

    grupos = [g[target].values for _, g in sub.groupby(cat_col)]
    grupos = [g for g in grupos if len(g) > 0]
    if len(grupos) >= 2:
        stat, p = kruskal(*grupos)
        print(f"  Kruskal-Wallis sobre {target}: H={stat:.3f} (p={p:.3g})")


def detalle_binaria_vs_target(df, bin_col, target, titulo):
    sub = df[[bin_col, target]].dropna()
    if len(sub) < 10 or sub[bin_col].nunique() != 2:
        return
    orden = sorted(sub[bin_col].unique(), key=lambda x: str(x))
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, val in zip(axes, orden):
        vals = sub.loc[sub[bin_col] == val, target]
        ax.boxplot(vals, patch_artist=True, showmeans=True)
        ax.set_title(f"{bin_col} = {val}")
        ax.set_ylabel(target)
    fig.suptitle(f"{titulo}: {target} por {bin_col}")
    plt.tight_layout()
    plt.show()

    g0 = sub.loc[sub[bin_col] == orden[0], target]
    g1 = sub.loc[sub[bin_col] == orden[1], target]
    stat, p = mannwhitneyu(g0, g1, alternative="two-sided")
    diff_med = g1.median() - g0.median()
    print(
        f"  Mann-Whitney U={stat:.0f} (p={p:.3g}) | "
        f"diferencia medianas ({orden[1]} - {orden[0]}) = {diff_med:.2f}"
    )


def detalle_categorica_categorica(df, c1, c2, titulo, max_cat=10):
    sub = df[[c1, c2]].dropna()
    if len(sub) < 10:
        return
    if sub[c1].nunique() > max_cat:
        sub = sub[sub[c1].isin(_top_categorias(sub[c1], max_cat))]
    if sub[c2].nunique() > max_cat:
        sub = sub[sub[c2].isin(_top_categorias(sub[c2], max_cat))]
    if sub[c1].nunique() < 2 or sub[c2].nunique() < 2:
        return

    v = cramers_v(sub[c1], sub[c2])
    ct = pd.crosstab(sub[c1], sub[c2])
    print(f"  Cramer V({c1}, {c2}) = {v:.3f}")
    print(ct)

    fig, ax = plt.subplots(figsize=(max(6, ct.shape[1] * 0.6), max(4, ct.shape[0] * 0.35)))
    sns.heatmap(ct, annot=True, fmt="d", cmap="Blues", ax=ax)
    ax.set_title(f"{titulo}: Cramer V = {v:.3f}")
    plt.tight_layout()
    plt.show()

    ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
    ct_pct.plot(kind="bar", stacked=True, figsize=(10, 4), colormap="tab20")
    plt.title(f"Barras 100% apiladas — {c1} vs {c2}")
    plt.ylabel("% dentro de fila")
    plt.xlabel(c1)
    plt.legend(title=c2, bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

    chi2, p, _, _ = chi2_contingency(ct)
    print(f"  Chi-cuadrado: chi2={chi2:.3f} (p={p:.3g})")


def detalle_num_con_hue_vs_target(df, x_col, hue_col, target, titulo, max_cat=8):
    sub = _muestra_plot(df[[x_col, hue_col, target]].dropna())
    if len(sub) < 20:
        return
    if sub[hue_col].nunique() > max_cat:
        sub = sub[sub[hue_col].isin(_top_categorias(sub[hue_col], max_cat))]

    fig, ax = plt.subplots(figsize=(7, 5))
    sns.scatterplot(data=sub, x=x_col, y=target, hue=hue_col, alpha=0.5, s=20, ax=ax)
    ax.set_title(f"{titulo}: {target} vs {x_col} (hue={hue_col})")
    plt.tight_layout()
    plt.show()

    g = sns.lmplot(
        data=sub,
        x=x_col,
        y=target,
        hue=hue_col,
        height=5,
        aspect=1.1,
        scatter_kws={"alpha": 0.35, "s": 15},
        line_kws={"linewidth": 1.5},
    )
    g.fig.suptitle(f"lmplot — {target} vs {x_col} por {hue_col}", y=1.02)
    plt.show()

    pairplot_numericas(
        sub,
        [x_col, target],
        hue_col,
        f"Pairplot con hue — {titulo}",
        max_cols=2,
    )


def seccion_detalle_vs_target(df, columnas, tipos, target, nombre_dataset):
    print("--- Nivel 2: cada feature vs objetivo ---")
    for col in columnas:
        t = tipos[col]
        print(f"\n>> {col} ({t}) vs {target}")
        if t == "numerica_continua":
            detalle_numerica_continua_vs_target(df, col, target, nombre_dataset)
        elif t in ("categorica", "numerica_discreta"):
            detalle_categorica_vs_target(df, col, target, nombre_dataset)
        elif t == "binaria":
            detalle_binaria_vs_target(df, col, target, nombre_dataset)


def seccion_numerica_por_categorica(df, columnas, tipos, nombre_dataset):
    nums = [c for c in columnas if tipos[c] == "numerica_continua"]
    cats = [c for c in columnas if tipos[c] in ("categorica", "numerica_discreta")]
    if not nums or not cats:
        print("(Sin pares numerica x categorica.)")
        return
    print("--- Numerica x categorica (distribucion de numerica por categoria) ---")
    for cat in cats:
        for num in nums:
            print(f"\n>> {num} por {cat}")
            detalle_numerica_vs_categorica(df, num, cat, nombre_dataset)


def seccion_num_hue_target(df, columnas, tipos, target, nombre_dataset, rank_target):
    nums = [c for c in columnas if tipos[c] == "numerica_continua"]
    cats = [c for c in columnas if tipos[c] in ("categorica", "numerica_discreta")]
    if not nums or not cats:
        return
    x_col = rank_target.loc[rank_target["feature"].isin(nums), "feature"]
    x_col = x_col.iloc[0] if len(x_col) else nums[0]
    hue_col = cats[0]
    print(f"--- Numerica x numerica x categorica (hue): {x_col}, {hue_col} vs {target} ---")
    detalle_num_con_hue_vs_target(df, x_col, hue_col, target, nombre_dataset)


def seccion_categorica_por_categorica(df, columnas, tipos, nombre_dataset, top_n=TOP_PARES_CAT_CAT):
    cats = [c for c in columnas if tipos[c] in ("categorica", "numerica_discreta")]
    pares = []
    for i, c1 in enumerate(cats):
        for c2 in cats[i + 1 :]:
            v = cramers_v(df[c1].dropna(), df[c2].dropna())
            if not np.isnan(v):
                pares.append((c1, c2, v))
    pares.sort(key=lambda x: x[2], reverse=True)
    if not pares:
        print("(Sin pares categorica x categorica.)")
        return
    print("--- Categorica x categorica ---")
    for c1, c2, v in pares[:top_n]:
        print(f"\n>> {c1} x {c2} (V={v:.3f})")
        detalle_categorica_categorica(df, c1, c2, nombre_dataset)


## Rentals — Bivariado (`rent_per_sqm`)


### Carga y preparacion


In [ ]:
from pathlib import Path
import pandas as pd

if "preparar_bivariado" not in globals():
    raise RuntimeError("Ejecuta primero la celda de FUNCIONES bivariado.")

_csv = Path("MOSCOW-DATASET/rentals.csv")
if not _csv.is_file():
    raise FileNotFoundError(f"No existe {_csv.resolve()}")

df_rentals = pd.read_csv(_csv)
print("df_rentals", df_rentals.shape)

mat_rentals, rank_rentals, rank_pares_rentals, tipos_rentals, cols_rentals = preparar_bivariado(
    df_rentals, target="rent_per_sqm", nombre_dataset="Rentals"
)


### Nivel 1 — Todas × todas (heatmap + ranking de pares)


In [ ]:
nivel1_todas_por_todas(
    mat_rentals, rank_pares_rentals, rank_rentals, target="rent_per_sqm", nombre_dataset="Rentals"
)


### Numericas — Pearson, Spearman y pairplot


In [ ]:
nums_rentals = nivel1_correlaciones_numericas(df_rentals, tipos_rentals, "Rentals")
hue_rentals = next(
    (c for c in cols_rentals if tipos_rentals.get(c) in ("categorica", "numerica_discreta")),
    None,
)
if len(nums_rentals) >= 2:
    pairplot_numericas(
        df_rentals, nums_rentals, hue_rentals or "rent_per_sqm",
        f"Pairplot numericas — Rentals",
    )


### Detalle vs `rent_per_sqm` (todas las features)


In [ ]:
seccion_detalle_vs_target(
    df_rentals, cols_rentals, tipos_rentals, target="rent_per_sqm", nombre_dataset="Rentals"
)


### Numerica × categorica


In [ ]:
seccion_numerica_por_categorica(df_rentals, cols_rentals, tipos_rentals, "Rentals")


### Numerica × categorica × objetivo (scatter/lmplot/pairplot con hue)


In [ ]:
seccion_num_hue_target(
    df_rentals, cols_rentals, tipos_rentals, target="rent_per_sqm", nombre_dataset="Rentals", rank_target=rank_rentals
)


### Categorica × categorica


In [ ]:
seccion_categorica_por_categorica(df_rentals, cols_rentals, tipos_rentals, "Rentals")


## Secondary market — Bivariado (`price_per_sqm`)


### Carga y preparacion


In [ ]:
from pathlib import Path
import pandas as pd

if "preparar_bivariado" not in globals():
    raise RuntimeError("Ejecuta primero la celda de FUNCIONES bivariado.")

_csv = Path("MOSCOW-DATASET/secondary_market.csv")
if not _csv.is_file():
    raise FileNotFoundError(f"No existe {_csv.resolve()}")

df_secondary = pd.read_csv(_csv)
print("df_secondary", df_secondary.shape)

mat_secondary, rank_secondary, rank_pares_secondary, tipos_secondary, cols_secondary = preparar_bivariado(
    df_secondary, target="price_per_sqm", nombre_dataset="Secondary market"
)


### Nivel 1 — Todas × todas (heatmap + ranking de pares)


In [ ]:
nivel1_todas_por_todas(
    mat_secondary, rank_pares_secondary, rank_secondary, target="price_per_sqm", nombre_dataset="Secondary market"
)


### Numericas — Pearson, Spearman y pairplot


In [ ]:
nums_secondary = nivel1_correlaciones_numericas(df_secondary, tipos_secondary, "Secondary market")
hue_secondary = next(
    (c for c in cols_secondary if tipos_secondary.get(c) in ("categorica", "numerica_discreta")),
    None,
)
if len(nums_secondary) >= 2:
    pairplot_numericas(
        df_secondary, nums_secondary, hue_secondary or "price_per_sqm",
        f"Pairplot numericas — Secondary market",
    )


### Detalle vs `price_per_sqm` (todas las features)


In [ ]:
seccion_detalle_vs_target(
    df_secondary, cols_secondary, tipos_secondary, target="price_per_sqm", nombre_dataset="Secondary market"
)


### Numerica × categorica


In [ ]:
seccion_numerica_por_categorica(df_secondary, cols_secondary, tipos_secondary, "Secondary market")


### Numerica × categorica × objetivo (scatter/lmplot/pairplot con hue)


In [ ]:
seccion_num_hue_target(
    df_secondary, cols_secondary, tipos_secondary, target="price_per_sqm", nombre_dataset="Secondary market", rank_target=rank_secondary
)


### Categorica × categorica


In [ ]:
seccion_categorica_por_categorica(df_secondary, cols_secondary, tipos_secondary, "Secondary market")


## New builds — Bivariado (`price_per_sqm`)


### Carga y preparacion


In [ ]:
from pathlib import Path
import pandas as pd

if "preparar_bivariado" not in globals():
    raise RuntimeError("Ejecuta primero la celda de FUNCIONES bivariado.")

_csv = Path("MOSCOW-DATASET/new_builds.csv")
if not _csv.is_file():
    raise FileNotFoundError(f"No existe {_csv.resolve()}")

df_newbuilds = pd.read_csv(_csv)
print("df_newbuilds", df_newbuilds.shape)

mat_newbuilds, rank_newbuilds, rank_pares_newbuilds, tipos_newbuilds, cols_newbuilds = preparar_bivariado(
    df_newbuilds, target="price_per_sqm", nombre_dataset="New builds"
)


### Nivel 1 — Todas × todas (heatmap + ranking de pares)


In [ ]:
nivel1_todas_por_todas(
    mat_newbuilds, rank_pares_newbuilds, rank_newbuilds, target="price_per_sqm", nombre_dataset="New builds"
)


### Numericas — Pearson, Spearman y pairplot


In [ ]:
nums_newbuilds = nivel1_correlaciones_numericas(df_newbuilds, tipos_newbuilds, "New builds")
hue_newbuilds = next(
    (c for c in cols_newbuilds if tipos_newbuilds.get(c) in ("categorica", "numerica_discreta")),
    None,
)
if len(nums_newbuilds) >= 2:
    pairplot_numericas(
        df_newbuilds, nums_newbuilds, hue_newbuilds or "price_per_sqm",
        f"Pairplot numericas — New builds",
    )


### Detalle vs `price_per_sqm` (todas las features)


In [ ]:
seccion_detalle_vs_target(
    df_newbuilds, cols_newbuilds, tipos_newbuilds, target="price_per_sqm", nombre_dataset="New builds"
)


### Numerica × categorica


In [ ]:
seccion_numerica_por_categorica(df_newbuilds, cols_newbuilds, tipos_newbuilds, "New builds")


### Numerica × categorica × objetivo (scatter/lmplot/pairplot con hue)


In [ ]:
seccion_num_hue_target(
    df_newbuilds, cols_newbuilds, tipos_newbuilds, target="price_per_sqm", nombre_dataset="New builds", rank_target=rank_newbuilds
)


### Categorica × categorica


In [ ]:
seccion_categorica_por_categorica(df_newbuilds, cols_newbuilds, tipos_newbuilds, "New builds")
